In [1]:
# %% [markdown]
# BEACON — Step 11: Model Training (XGBoost + Optuna)
#
# Run this script TWICE — once per stream — by changing STREAM below.
# It picks up directly from processed_final/ (Steps 1-10 output).
#
# Install once: pip install xgboost optuna joblib scikit-learn seaborn matplotlib

# %%
import os
import numpy as np
import pandas as pd

# ============================================================
# CONFIG — EDIT THIS LINE to switch streams
# ============================================================
STREAM = "network"  # "network" or "memory"

FINAL_DIR = r"D:\Malware Dataset\processed_final"
MODEL_DIR = r"D:\Malware Dataset\models"
os.makedirs(MODEL_DIR, exist_ok=True)

PREFIX = "net" if STREAM == "network" else "mem"

# is_synthetic is metadata only (Memory/Exploit augmentation flag) — never a
# model feature. label and sample_id are excluded the same way on both streams.
NON_FEATURE_COLS = ["label", "sample_id", "is_synthetic"]

N_OPTUNA_TRIALS = 30  # increase if you have time/compute to spare

# %%
# ============================================================
# Load the validated train/test partitions and saved sample weights
# ============================================================
train_df = pd.read_parquet(os.path.join(FINAL_DIR, f"{PREFIX}_train.parquet"))
test_df = pd.read_parquet(os.path.join(FINAL_DIR, f"{PREFIX}_test.parquet"))
full_train_weights = np.load(os.path.join(FINAL_DIR, f"{PREFIX}_sample_weights.npy"))

# Use the intersection of train/test columns as the feature set — robust to
# is_synthetic being present in train only (expected on the Memory stream).
feature_cols = [c for c in train_df.columns if c not in NON_FEATURE_COLS and c in test_df.columns]

print(f"Stream: {STREAM}")
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")
print(f"Feature columns used: {len(feature_cols)}")

# %%
# ============================================================
# Label encoding
# ============================================================
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_full = le.fit_transform(train_df["label"])
y_test = le.transform(test_df["label"])

X_train_full = train_df[feature_cols]
X_test = test_df[feature_cols]
N_CLASSES = len(le.classes_)

print("Classes:", list(le.classes_))

# %%
# ============================================================
# Inner grouped split — for Optuna tuning ONLY, separate from the
# outer test set saved earlier. Still grouped by sample_id, same
# leakage-avoidance principle as the original Step 5 split.
# ============================================================
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight

sgkf_inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
inner_train_idx, inner_val_idx = next(
    sgkf_inner.split(X_train_full, y_train_full, groups=train_df["sample_id"])
)

X_tr, X_val = X_train_full.iloc[inner_train_idx], X_train_full.iloc[inner_val_idx]
y_tr, y_val = y_train_full[inner_train_idx], y_train_full[inner_val_idx]

# Recompute balanced weights on this inner subset specifically — the saved
# full_train_weights correspond to the entire training set and aren't valid
# for this smaller inner split.
w_tr = compute_sample_weight("balanced", y_tr)
w_val = compute_sample_weight("balanced", y_val)

print(f"Inner train: {X_tr.shape}, Inner val: {X_val.shape}")

# %%
# ============================================================
# Optuna hyperparameter search
# ============================================================
import optuna
import xgboost as xgb
from sklearn.metrics import f1_score

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "objective": "multi:softprob",
        "num_class": N_CLASSES,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
        "early_stopping_rounds": 30,
    }

    model = xgb.XGBClassifier(**params)
    model.fit(
        X_tr, y_tr, sample_weight=w_tr,
        eval_set=[(X_val, y_val)],
        sample_weight_eval_set=[w_val],
        verbose=False,
    )
    preds = model.predict(X_val)
    # Macro F1: every class weighted equally regardless of size — matches
    # the project's stated target metric and doesn't let Virus/Backdoor-sized
    # classes dominate the tuning objective.
    return f1_score(y_val, preds, average="macro")

study = optuna.create_study(direction="maximize", study_name=f"beacon_{STREAM}")
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print("\nBest inner-validation macro F1:", study.best_value)
print("Best params:", study.best_params)

# %%
# ============================================================
# Train the FINAL model on the FULL training set, using the
# original saved sample weights (not the inner-split ones above)
# ============================================================
best_params = study.best_params.copy()
best_params.update({
    "objective": "multi:softprob",
    "num_class": N_CLASSES,
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
})

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train_full, y_train_full, sample_weight=full_train_weights, verbose=False)

print("Final model trained on the full training set.")

# %%
# ============================================================
# Evaluate on the held-out test set
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = final_model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3))

macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"Test macro F1: {macro_f1:.4f}")

if STREAM == "memory" and "Exploit" in le.classes_:
    print("\nNote: Exploit's training data is ~90% synthetic (cluster-based "
          "SMOTE, Step 10). Treat its test performance above as lower-confidence "
          "relative to the other 8 categories, per the documented caveat.")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"{STREAM.capitalize()} Stream — Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, f"{STREAM}_confusion_matrix.png"), dpi=150)
plt.show()

# %%
# ============================================================
# Save the model, label encoder, and Optuna study for this stream
# ============================================================
import joblib

joblib.dump(final_model, os.path.join(MODEL_DIR, f"{STREAM}_xgboost_model.joblib"))
joblib.dump(le, os.path.join(MODEL_DIR, f"{STREAM}_label_encoder.joblib"))
joblib.dump(study, os.path.join(MODEL_DIR, f"{STREAM}_optuna_study.joblib"))

print(f"\nSaved model, label encoder, and Optuna study for the {STREAM} stream to:")
print(MODEL_DIR)
print("\nSet STREAM to the other value and re-run this script for the second model.")

Stream: network
Train shape: (517217, 188), Test shape: (129358, 188)
Feature columns used: 186
Classes: ['Backdoor', 'Benign', 'Exploit', 'HackTool', 'Hoax', 'Rootkit', 'Trojan', 'Virus', 'Worm']
Inner train: (413826, 186), Inner val: (103391, 186)


[I 2026-07-27 13:18:39,600] A new study created in memory with name: beacon_network


  0%|          | 0/30 [00:00<?, ?it/s]

[W 2026-07-27 13:18:39,694] Trial 0 failed with parameters: {'n_estimators': 450, 'max_depth': 8, 'learning_rate': 0.027034226871804953, 'subsample': 0.8183860250091435, 'colsample_bytree': 0.8279523560291163} because of the following error: ValueError('DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:flow_id: str, timestamp: str, src_ip: str, dst_ip: str, protocol: str, payload_bytes_skewness: str, payload_bytes_cov: str, fwd_payload_bytes_skewness: str, fwd_payload_bytes_cov: str, bwd_payload_bytes_skewness: str, bwd_payload_bytes_cov: str, skewness_header_bytes: str, fwd_skewness_header_bytes: str, bwd_skewness_header_bytes: str, fwd_segment_size_skewness: str, bwd_segment_size_skewness: str, segment_size_skewness: str, active_skewness: str, idle_skewness: str, packets_IAT_skewness: str, fwd_packets_IAT_skewness: str, fwd_packets_IAT_cov: s

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:flow_id: str, timestamp: str, src_ip: str, dst_ip: str, protocol: str, payload_bytes_skewness: str, payload_bytes_cov: str, fwd_payload_bytes_skewness: str, fwd_payload_bytes_cov: str, bwd_payload_bytes_skewness: str, bwd_payload_bytes_cov: str, skewness_header_bytes: str, fwd_skewness_header_bytes: str, bwd_skewness_header_bytes: str, fwd_segment_size_skewness: str, bwd_segment_size_skewness: str, segment_size_skewness: str, active_skewness: str, idle_skewness: str, packets_IAT_skewness: str, fwd_packets_IAT_skewness: str, fwd_packets_IAT_cov: str, bwd_packets_IAT_skewness: str, bwd_packets_IAT_cov: str, delta_start: str, handshake_duration: str, skewness_packets_delta_time: str, skewness_bwd_packets_delta_time: str, skewness_fwd_packets_delta_time: str, cov_fwd_packets_delta_time: 